# Determine And Validate The RNA Preprocessing Rules

## Purpose

**This notebook evaluates preprocessing choices for RNA expression data after EDA filtering.**

### Objectives:
- Examine RNA expression distributions
- Evaluate low-expression filtering strategies
- Evaluate variance-based feature filtering
- Assess scaling effects on gene expression features
- Prototype preprocessing steps before implementation in the pipeline

### Workflow

1. Load filtered RNA expression dataset  
   - Dataset already passed validity filtering during EDA.

2. Load split IDs and partition the dataset  
   - Create `X_train`, `X_val`, `X_test` using the saved split.

**Subsequent exploratory steps use the training data only.**

3. Inspect expression distributions  
   - Examine overall gene expression distributions across samples.  
   - Identify potential skew, heavy tails, or extreme values.

4. Evaluate low-expression filtering  
   - Explore thresholds for removing genes with consistently low expression.  
   - Assess impact on feature count and expression distribution.

5. Evaluate variance filtering  
   - Identify genes with near-zero variance across samples.  
   - Assess potential thresholds for removing uninformative genes.

6. Assess scaling effects  
   - Evaluate the effect of standardization on expression features.  
   - Confirm scaling behavior across genes.

7. Define preprocessing rules  
   - Select deterministic filtering thresholds and scaling strategy.

8. Fit preprocessing parameters on the training set only  
   - Compute filtering thresholds and scaling parameters using `X_train`.

9. Apply preprocessing to all splits  
   - Transform train, validation, and test using parameters learned from the training set.

10. Validate outputs  
   - Confirm expected feature counts after filtering  
   - Verify row counts match split sizes  
   - Verify no ID drift or leakage

11. Save preprocessing parameters  
   - Output filtering thresholds and scaling parameters for use in the preprocessing pipeline.

12. Test preprocessing module

In [4]:
from pathlib import Path

import pandas as pd

## 1. Load filtered RNA expression dataset  
   - Dataset already passed validity filtering during EDA.

In [9]:
# Load the filtered RNA cohort by keeping only the saved sample IDs.
rna_path = Path("../data/raw/TCGA-BRCA.star_tpm.tsv.gz")
sample_ids_df = pd.read_csv("../data/interim/sample_ids_filtered.csv")

sample_ids = sample_ids_df["sample"].astype(str)
rna_df = pd.read_csv(rna_path, sep="\t")

if "Ensembl_ID" not in rna_df.columns:
    raise KeyError("Expected 'Ensembl_ID' column in RNA expression table")

missing_sample_ids = sorted(set(sample_ids) - set(rna_df.columns))
if missing_sample_ids:
    raise KeyError(f"Filtered sample IDs missing from RNA table: {missing_sample_ids[:5]}")

# Keep genes as features and samples as rows for downstream splitting.
rna_df = (
    rna_df.set_index("Ensembl_ID")
    .loc[:, sample_ids]
    .T
)
rna_df.index.name = "sample"

print(f"RNA shape (samples x genes): {rna_df.shape}")
display(rna_df.head(), rna_df.tail())

RNA shape (samples x genes): (290, 60660)


Ensembl_ID,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-3C-AAAU-01A,3.407747,0.125387,6.712645,3.358185,2.264056,2.563549,3.112116,4.795622,3.737081,5.948982,...,0.0,0.000000,0.601316,0.0,0.0,0.000000,4.962211,0.0,0.179256,1.284100
TCGA-3C-AALI-01A,3.228680,0.167486,7.107211,5.052007,3.080419,3.514173,4.391307,5.583462,4.177344,4.947381,...,0.0,1.330329,0.806860,0.0,0.0,0.032524,3.900162,0.0,0.117695,1.779932
TCGA-A1-A0SK-01A,6.590713,0.000000,7.653724,3.384616,3.907929,1.733311,2.307370,5.456110,4.120941,6.433409,...,0.0,0.000000,0.146655,0.0,0.0,0.000000,3.574804,0.0,0.030124,0.682753
TCGA-A2-A04N-01A,5.665933,0.288418,6.524913,3.503972,2.447156,2.538414,4.421715,5.488293,4.331956,5.614557,...,0.0,0.000000,0.544485,0.0,0.0,0.000000,3.993702,0.0,0.029135,0.979733
TCGA-A2-A04Q-01A,4.769962,0.195348,6.077078,3.461529,2.534908,4.497389,4.031960,5.251401,3.198227,5.382142,...,0.0,0.000000,0.144177,0.0,0.0,0.000000,3.404794,0.0,0.038015,0.665302


Ensembl_ID,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-PE-A5DD-01A,4.593270,1.214000,6.272486,4.041217,3.159532,3.589260,4.791069,5.397139,3.373913,5.326721,...,0.0,0.0,0.988703,0.0,0.0,0.000000,2.385652,0.0,0.196355,1.047329
TCGA-PE-A5DE-01A,3.057242,0.639325,6.935899,4.346141,3.010064,4.666876,4.787088,4.581002,3.403936,5.482932,...,0.0,0.0,0.451646,0.0,0.0,0.000000,3.220748,0.0,0.108089,0.985355
TCGA-V7-A7HQ-01A,5.278453,0.325847,5.477891,2.152183,0.690551,1.788686,2.869556,5.829055,3.082890,4.157084,...,0.0,0.0,0.091666,0.0,0.0,0.000000,2.764537,0.0,0.044184,1.893751
TCGA-Z7-A8R5-01A,5.526410,0.674551,6.273221,3.354353,1.419431,4.602688,4.369222,5.659548,2.981177,4.529908,...,0.0,0.0,0.324926,0.0,0.0,0.028711,2.736843,0.0,0.058801,1.280006
TCGA-Z7-A8R6-01A,5.003499,0.339023,6.923104,3.757770,3.868114,3.141400,2.935064,5.057849,3.228295,5.447358,...,0.0,0.0,0.485736,0.0,0.0,0.000000,3.481661,0.0,0.028993,1.536501


## 2. Load split IDs and partition the dataset  
   - Create `X_train`, `X_val`, `X_test` using the saved split.

In [10]:
# Load saved split IDs and partition the RNA dataset by sample.
split_dir = Path("../data/processed/splits/os_seed42_v15_t15")

train_ids_df = pd.read_csv(split_dir / "train_ids.csv")
val_ids_df = pd.read_csv(split_dir / "val_ids.csv")
test_ids_df = pd.read_csv(split_dir / "test_ids.csv")

train_ids = train_ids_df["sample"].astype(str)
val_ids = val_ids_df["sample"].astype(str)
test_ids = test_ids_df["sample"].astype(str)

missing_ids = sorted((set(train_ids) | set(val_ids) | set(test_ids)) - set(rna_df.index))
if missing_ids:
    raise KeyError(f"Split sample IDs missing from rna_df: {missing_ids[:5]}")

# Keep split order fixed so downstream comparisons stay aligned across modalities.
X_train_df = rna_df.loc[train_ids].copy()
X_val_df = rna_df.loc[val_ids].copy()
X_test_df = rna_df.loc[test_ids].copy()

print(f"train: {X_train_df.shape}")
print(f"val:   {X_val_df.shape}")
print(f"test:  {X_test_df.shape}")
display(X_train_df.head(), X_train_df.tail())


train: (203, 60660)
val:   (43, 60660)
test:  (44, 60660)


Ensembl_ID,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-AR-A1AN-01A,4.136962,1.340391,6.126083,3.469365,2.488952,3.092038,5.305310,5.972906,4.263613,5.541366,...,0.0,0.0,0.242572,0.0,0.000000,0.0,4.139371,0.0,0.008630,1.517981
TCGA-B6-A2IU-01A,4.087047,1.623258,6.016039,5.286083,3.669526,2.460087,3.389126,4.965530,4.565177,5.943427,...,0.0,0.0,0.318461,0.0,0.000000,0.0,4.117537,0.0,0.056445,0.543001
TCGA-C8-A12Q-01A,5.143761,1.293017,7.245631,4.338146,3.223716,3.880284,5.129254,5.493731,4.278594,5.370028,...,0.0,0.0,0.318577,0.0,0.000000,0.0,4.181667,0.0,0.065986,0.521955
TCGA-BH-A0E9-01B,6.761509,1.384271,6.749827,4.912942,3.892906,3.541341,5.536118,5.905336,3.981761,6.115972,...,0.0,0.0,0.466027,0.0,1.079498,0.0,4.501280,0.0,0.098420,1.036327
TCGA-A2-A0ST-01A,5.294275,0.854474,6.285212,3.034726,2.585732,4.877587,4.336462,5.668272,3.225260,5.522232,...,0.0,0.0,0.207518,0.0,0.000000,0.0,2.806901,0.0,0.053389,0.849999


Ensembl_ID,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
sample,,,,,,,,,,,,,,,,,,,,,
TCGA-GM-A2DM-01A,6.403268,0.070939,6.505716,3.298204,1.804921,3.107119,2.665802,6.461699,5.408236,5.437391,...,0.0,0.0,0.226262,0.0,0.000000,0.0,3.883709,0.0,0.076149,1.135206
TCGA-AR-A2LE-01A,6.643308,0.384492,5.785514,4.133588,2.392427,2.037031,4.085059,5.312110,4.510905,5.224361,...,0.0,0.0,0.526770,0.0,0.000000,0.0,3.897240,0.0,0.066124,0.655352
TCGA-E2-A1LK-01A,5.512546,0.000000,7.019724,3.237120,2.695437,3.149845,1.664528,6.021271,3.353450,5.099480,...,0.0,0.0,0.206893,0.0,0.000000,0.0,4.087870,0.0,0.013069,0.792855
TCGA-E9-A1RB-01A,4.570669,0.161436,7.336596,3.950067,4.061197,2.840040,4.378331,5.724260,3.113200,4.963997,...,0.0,0.0,0.530570,0.0,0.000000,0.0,4.288329,0.0,0.038717,0.810649
TCGA-B6-A0I6-01A,5.543199,0.000000,7.148059,2.944334,2.548905,2.791147,4.206674,6.101925,3.907929,4.812565,...,0.0,0.0,0.061569,0.0,1.236768,0.0,3.271426,0.0,0.119024,0.198871


3. Inspect expression distributions  
   - Examine overall gene expression distributions across samples.  
   - Identify potential skew, heavy tails, or extreme values.



4. Evaluate low-expression filtering  
   - Explore thresholds for removing genes with consistently low expression.  
   - Assess impact on feature count and expression distribution.

5. Evaluate variance filtering  
   - Identify genes with near-zero variance across samples.  
   - Assess potential thresholds for removing uninformative genes.

6. Assess scaling effects  
   - Evaluate the effect of standardization on expression features.  
   - Confirm scaling behavior across genes.

7. Define preprocessing rules  
   - Select deterministic filtering thresholds and scaling strategy.

8. Fit preprocessing parameters on the training set only  
   - Compute filtering thresholds and scaling parameters using `X_train`.

9. Apply preprocessing to all splits  
   - Transform train, validation, and test using parameters learned from the training set.

10. Validate outputs  
   - Confirm expected feature counts after filtering  
   - Verify row counts match split sizes  
   - Verify no ID drift or leakage

11. Save preprocessing parameters  
   - Output filtering thresholds and scaling parameters for use in the preprocessing pipeline.

12. Test preprocessing module